# Interactive simulation checks: anemia screening

Verifies the IFA effect on hemoglobin, anemia-status assignment, hemoglobin-screening
coverage (baseline vs the `anemia_screening_vv` scenario), and that the hemoglobin test is
informative. Ported from the research portfolio VnV notebook
`model_18.3_interactive_simulation_anemia_screening`; updated to the current Engine
(`vivarium.engine`) API and to current model behavior.

Note: the source's `ifa_deleted_hemoglobin.exposure` / `first_anc_hemoglobin.exposure`
pipelines were removed, and the raw `hemoglobin_exposure` state column is not populated until
late in the timestep sequence -- so the IFA effect is expressed as IFA-vs-untreated
`hemoglobin.exposure`, and the sim is stepped to `delivery_facility` so the screening/anemia
columns are all populated. The exact test sensitivity/specificity targets (~0.85 / ~0.80) and
the precise hemoglobin measure the test screens are left for researchers to pin.

In [ ]:
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

import numpy as np
import pandas as pd
from pathlib import Path

import vivarium_gates_mncnh
from vivarium.engine import InteractiveContext
from vivarium.engine.framework.configuration import build_model_specification

In [ ]:
!pip list | grep vivarium

In [ ]:
SPEC_PATH = Path(vivarium_gates_mncnh.__file__).parent / "model_specifications/model_spec.yaml"
COLS = ["anc_attendance", "oral_iron_intervention", "hemoglobin_screening_coverage",
        "ferritin_screening_coverage", "tested_hemoglobin", "anemia_status_during_pregnancy",
        "hemoglobin.exposure"]

def build_sim(scenario=None):
    spec = build_model_specification(SPEC_PATH)
    del spec.configuration.observers
    spec.configuration.population.population_size = 20_000 * 10
    if scenario is not None:
        spec.configuration.intervention.scenario = scenario
    sim = InteractiveContext(spec)
    # Step through all ANC / screening events so the screening + anemia columns are populated.
    ev = sim._builder.time.simulation_event_name()
    while ev() != "delivery_facility":
        sim.step()
    return sim

def anemia_status_from(hb):
    return np.where(hb <= 70, "severe",
           np.where(hb <= 100, "moderate",
           np.where(hb <= 110, "mild", "not_anemic")))

In [ ]:
# Baseline scenario
sim = build_sim()
df = sim.get_population(COLS)
df[["anc_attendance", "oral_iron_intervention", "hemoglobin_screening_coverage",
    "tested_hemoglobin", "anemia_status_during_pregnancy"]].head()

## IFA raises hemoglobin

In [ ]:
# The IFA effect is applied within the `hemoglobin.exposure` pipeline, so IFA-treated
# simulants have a higher hemoglobin exposure than the untreated.
ifa = df.oral_iron_intervention == "ifa"
assert df.loc[ifa, "hemoglobin.exposure"].mean() > df.loc[~ifa, "hemoglobin.exposure"].mean(), \
    "IFA-treated simulants do not have higher hemoglobin than the untreated"

## Anemia status is ordered by hemoglobin

In [ ]:
# anemia_status_during_pregnancy is assigned to screened simulants from hemoglobin thresholds
# (severe <=70, moderate <=100, mild <=110, else not_anemic). Assert the assigned categories
# are ordered by mean hemoglobin.exposure.
known = ["severe", "moderate", "mild", "not_anemic"]
assigned = df[df.anemia_status_during_pregnancy.isin(known)]
assert len(assigned) > 0, "no simulants have an assigned anemia status"
order = assigned.groupby("anemia_status_during_pregnancy")["hemoglobin.exposure"].mean()
assert order["severe"] < order["moderate"] < order["mild"] < order["not_anemic"], \
    f"anemia-status categories not ordered by hemoglobin: {order.to_dict()}"

## Screening coverage: baseline

In [ ]:
# Baseline hemoglobin screening happens at the later-pregnancy ANC visit, so only attendees
# with a later visit are (partially) screened; first-trimester-only and no-ANC are not; and
# ferritin screening is off at baseline.
cov = df.groupby("anc_attendance").hemoglobin_screening_coverage.mean()
assert cov.loc["none"] == 0, "hemoglobin screening occurred among no-ANC simulants at baseline"
later = ["later_pregnancy_only", "first_trimester_and_later_pregnancy"]
assert ((cov.loc[later] > 0) & (cov.loc[later] < 1)).all(), \
    f"expected partial baseline screening among later-visit ANC attendees, got {cov.to_dict()}"
assert cov.get("first_trimester_only", 0) == 0, \
    "first-trimester-only attendees were screened at baseline (screening is at the later visit)"
assert (~df.ferritin_screening_coverage).all(), "ferritin screening should be off at baseline"

## The hemoglobin test is informative

In [ ]:
# Among screened simulants, the test should be well better than chance: most truly-low test
# low (sensitivity), most truly-adequate test adequate (specificity). Nominal targets are
# ~0.85 / ~0.80; truth is taken from hemoglobin.exposure (< 100 g/L). Researchers can tighten
# to exact targets and to whatever hemoglobin measure the test actually screens.
tested = df[df.tested_hemoglobin != "not_tested"].copy()
tested["truth"] = np.where(tested["hemoglobin.exposure"] < 100, "low", "adequate")
sens = (tested.loc[tested.truth == "low", "tested_hemoglobin"] == "low").mean()
spec = (tested.loc[tested.truth == "adequate", "tested_hemoglobin"] == "adequate").mean()
assert sens > 0.7, f"hemoglobin-test sensitivity {sens:.3f} unexpectedly low (target ~0.85)"
assert spec > 0.7, f"hemoglobin-test specificity {spec:.3f} unexpectedly low (target ~0.80)"

## Screening coverage: `anemia_screening_vv` scale-up scenario

In [ ]:
# In the anemia-screening VnV scenario, later-pregnancy ANC attendees are all screened for
# hemoglobin and no-ANC simulants are still never screened.
vv = build_sim(scenario="anemia_screening_vv")
vv_cov = vv.get_population(["anc_attendance", "hemoglobin_screening_coverage"]) \
    .groupby("anc_attendance").hemoglobin_screening_coverage.mean()
assert vv_cov.loc["none"] == 0, "hemoglobin screening among no-ANC simulants in anemia_screening_vv"
later = ["later_pregnancy_only", "first_trimester_and_later_pregnancy"]
assert (vv_cov.loc[later] == 1).all(), \
    f"expected 100% hemoglobin screening at later-visit ANC in anemia_screening_vv, got {vv_cov.to_dict()}"